In [ ]:
# ============================================================
# LANGCHAIN MASTER REVISION
# Model → Structured Output → Tools → Agent → Stream/Batch
# → Middleware → HITL → Summarization
#
# Models:
#   Ollama: qwen2.5-coder:3b
#   Gemini: gemini-2.5-flash-lite
# ============================================================


# ------------------------------------------------------------
# 0. INSTALL
# ------------------------------------------------------------
#
# pip install -U langchain langchain-ollama langchain-google-genai
#
# For HITL/checkpointing:
#
# pip install -U langgraph
#
# Ollama:
#
# ollama pull qwen2.5-coder:3b
#
# Gemini:
# Set GOOGLE_API_KEY in your environment.
#
# ------------------------------------------------------------


from pydantic import BaseModel, Field

from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI

from langchain.agents import create_agent
from langchain.tools import tool

from langchain.agents.middleware import (
    HumanInTheLoopMiddleware,
    SummarizationMiddleware,
)

from langgraph.checkpoint.memory import InMemorySaver
from langgraph.types import Command


# ============================================================
# 1. MODELS
# ============================================================

ollama_model = ChatOllama(
    model="qwen2.5-coder:3b",
    temperature=0,
)

gemini_model = ChatGoogleGenerativeAI(
    model="gemini-2.5-flash-lite",
    temperature=0,
)


# ============================================================
# 2. MODEL.INVOKE()
# ============================================================

# This is the basic LLM call.
#
# Input:
#     text/messages
#
# Output:
#     AIMessage
#
# No agent.
# No tools.
# No agent loop.

response = gemini_model.invoke(
    "Explain gradient descent in one sentence."
)

print(response.content)


# Same thing with Ollama:

response = ollama_model.invoke(
    "Explain gradient descent in one sentence."
)

print(response.content)


# ============================================================
# 3. MODEL WITH STRUCTURED OUTPUT
# ============================================================

# Pydantic schema

class Weather(BaseModel):
    city: str = Field(description="City name")
    temperature: float = Field(description="Temperature in Celsius")
    condition: str = Field(description="Weather condition")


# Convert the model into a structured-output model.

structured_gemini = gemini_model.with_structured_output(Weather)


response = structured_gemini.invoke(
    "The weather in Kolkata is 31 degrees Celsius and sunny."
)

print(response)

# Result is a Pydantic object:
#
# Weather(
#     city="Kolkata",
#     temperature=31,
#     condition="sunny"
# )


# IMPORTANT:
#
# model.with_structured_output(...)
#
# means:
#
#     "I want the MODEL'S response to follow this schema."
#
# This is NOT an agent.


# ============================================================
# 4. TOOL
# ============================================================

@tool
def get_weather(city: str):
    """Get the weather for a city."""
    print(">>> TOOL EXECUTED")
    return f"It is sunny in {city} today."


# The decorator isn't strictly necessary for simple functions.
#
# This also works:
#
# def get_weather(city: str):
#     """Get the weather for a city."""
#     return f"It is sunny in {city} today."
#
# create_agent(..., tools=[get_weather])
#
# LangChain can convert the function into a tool.
#
# @tool simply makes the conversion explicit and gives you
# a LangChain tool object.


# ============================================================
# 5. MODEL TOOL CALLING — bind_tools()
# ============================================================

model_with_tools = gemini_model.bind_tools(
    [get_weather]
)


response = model_with_tools.invoke(
    "What's the weather in Kolkata?"
)


print(response)

print(response.tool_calls)


# IMPORTANT:
#
# bind_tools()
#
# gives the MODEL the ability to REQUEST a tool.
#
# It does NOT create the agent loop.
#
#
# MODEL:
#
#     user
#       ↓
#     LLM
#       ↓
#     tool call
#
# END
#
# You would have to execute the tool yourself and send the
# ToolMessage back to the model.
#
#
# create_agent()
#
# does this loop automatically.


# ============================================================
# 6. AGENT
# ============================================================

agent = create_agent(
    model=gemini_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)


result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?",
            }
        ]
    }
)


print(result)


# The agent internally does:
#
# USER
#   ↓
# MODEL
#   ↓
# "I should call get_weather"
#   ↓
# TOOL
#   ↓
# get_weather("Kolkata")
#   ↓
# TOOL RESULT
#   ↓
# MODEL
#   ↓
# FINAL ANSWER
#
#
# This is the major difference between:
#
# model.bind_tools()
#
# and
#
# create_agent(..., tools=[...])


# ============================================================
# 7. AGENT + PYDANTIC STRUCTURED OUTPUT
# ============================================================

class WeatherAnswer(BaseModel):
    city: str
    weather: str
    recommendation: str


structured_agent = create_agent(
    model=gemini_model,
    tools=[get_weather],

    # IMPORTANT:
    # This is how an AGENT gets structured output.
    #
    # NOT:
    #     agent.with_structured_output(...)
    #
    # Instead:
    #     response_format=...
    #
    response_format=WeatherAnswer,
)


result = structured_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Tell me the weather in Kolkata and what I should do today.",
            }
        ]
    }
)


print(result["structured_response"])


# Result:
#
# WeatherAnswer(
#     city="Kolkata",
#     weather="Sunny",
#     recommendation="..."
# )
#
#
# Remember:
#
# MODEL:
#
#     model.with_structured_output(Weather)
#
#
# AGENT:
#
#     create_agent(
#         ...
#         response_format=Weather
#     )
#
#
# Agent structured output is returned in:
#
#     result["structured_response"]


# ============================================================
# 8. AGENT.INVOKE()
# ============================================================

result = agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Hello!",
            }
        ]
    }
)

print(result)


# invoke()
#
#     one input
#     one complete execution
#     returns final state


# ============================================================
# 9. AGENT.STREAM()
# ============================================================

# stream() lets you observe execution as it happens.

for chunk in agent.stream(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?",
            }
        ]
    },
    stream_mode="updates",
):
    print(chunk)


# Conceptually:
#
#     MODEL UPDATE
#          ↓
#     TOOL UPDATE
#          ↓
#     MODEL UPDATE
#
#
# Another useful mode is "messages":
#
# for chunk in agent.stream(
#     {...},
#     stream_mode="messages",
# ):
#     print(chunk)
#
# This is useful when you want token/message streaming.


# ============================================================
# 10. AGENT.BATCH()
# ============================================================

results = agent.batch(
    [
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in Kolkata?",
                }
            ]
        },
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in Delhi?",
                }
            ]
        },
        {
            "messages": [
                {
                    "role": "user",
                    "content": "What's the weather in Mumbai?",
                }
            ]
        },
    ]
)


for result in results:
    print(result["messages"][-1].content)


# Remember:
#
# invoke()
#     one input
#
# batch()
#     many inputs
#
# stream()
#     execution streamed as it happens


# ============================================================
# 11. SAME AGENT IDEA WITH OLLAMA
# ============================================================

ollama_agent = create_agent(
    model=ollama_model,
    tools=[get_weather],
    system_prompt="You are a helpful assistant.",
)


result = ollama_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "What's the weather in Kolkata?",
            }
        ]
    }
)


print(result)


# IMPORTANT:
#
# This is the SAME LangChain agent API.
#
# The only thing that changed is:
#
#     gemini_model
#
# versus
#
#     ollama_model
#
#
# But tool-calling reliability depends on whether the
# particular model/provider properly supports tool calling.
#
# Your Qwen experiment demonstrated this:
#
#     content = '{"name": "get_weather", ...}'
#
#     tool_calls = []
#
# That means the model generated JSON-like TEXT instead
# of a proper structured tool call.


# ============================================================
# 12. HUMAN-IN-THE-LOOP
# ============================================================

# HITL means:
#
#     MODEL proposes dangerous/actionable tool call
#                ↓
#           PAUSE EXECUTION
#                ↓
#           HUMAN REVIEWS
#                ↓
#       approve / edit / reject
#                ↓
#           execution continues
#
#
# You NEED a checkpointer because the agent must remember
# its state while execution is paused.


@tool
def send_email(
    to: str,
    subject: str,
    body: str,
):
    """Send an email."""
    print(">>> EMAIL TOOL EXECUTED")
    return f"Email sent to {to}"


hitl_agent = create_agent(
    model=gemini_model,

    tools=[
        get_weather,
        send_email,
    ],

    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                # All HITL decisions allowed:
                #
                # approve
                # edit
                # reject
                #
                "send_email": True,

                # Weather is safe.
                # No human approval.
                "get_weather": False,
            }
        )
    ],

    # REQUIRED for HITL
    checkpointer=InMemorySaver(),
)


# Thread ID identifies the paused/resumable conversation.

config = {
    "configurable": {
        "thread_id": "demo-thread-1"
    }
}


# Start the agent.

result = hitl_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": (
                    "Send an email to boss@example.com "
                    "saying I will be late today."
                ),
            }
        ]
    },
    config=config,
    version="v2",
)


print(result)


# At this point the agent can INTERRUPT before executing
# send_email.


# ============================================================
# 13. HITL ACTION #1 — APPROVE
# ============================================================

# Human says:
#
# "Yes, execute exactly what the agent proposed."


result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "approve"
                }
            ]
        }
    ),
    config=config,
    version="v2",
)


# APPROVE:
#
#     model proposed:
#
#         send_email(
#             to="boss@example.com",
#             ...
#         )
#
#     human:
#
#         approve
#
#     ↓
#
#     EXACT SAME TOOL CALL EXECUTES


# ============================================================
# 14. HITL ACTION #2 — EDIT
# ============================================================

# Imagine the model proposed:
#
#     to="boss@example.com"
#
# Human wants:
#
#     to="manager@example.com"
#
# Instead of rejecting everything, modify the action.


result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "edit",
                    "edited_action": {
                        "name": "send_email",
                        "args": {
                            "to": "manager@example.com",
                            "subject": "Running late",
                            "body": "I will be late today.",
                        },
                    },
                }
            ]
        }
    ),
    config=config,
    version="v2",
)


# EDIT:
#
#     model proposed action
#            ↓
#     human modifies arguments
#            ↓
#     edited tool call executes


# ============================================================
# 15. HITL ACTION #3 — REJECT
# ============================================================

# Human says:
#
# "No. Don't send this email."


result = hitl_agent.invoke(
    Command(
        resume={
            "decisions": [
                {
                    "type": "reject",
                    "message": (
                        "Do not send this email. "
                        "Ask me before sending anything."
                    ),
                }
            ]
        }
    ),
    config=config,
    version="v2",
)


# REJECT:
#
#     tool DOES NOT execute.
#
#     Instead the rejection + feedback is passed back
#     into the agent conversation.
#
#     The model can then decide what to do next.


# ============================================================
# HITL — THE 3 CORE ACTIONS
# ============================================================
#
# APPROVE
#     Execute exactly what the model proposed.
#
# EDIT
#     Modify the proposed tool call, then execute the
#     modified version.
#
# REJECT
#     Don't execute it.
#     Give feedback/reason to the model.
#
#
# Current LangChain also supports "respond" for special
# ask-user style tools, where the human supplies the tool
# result directly instead of executing the tool.
#
# But the three actions you should remember for ordinary
# tool-approval workflows are:
#
#     APPROVE
#     EDIT
#     REJECT


# ============================================================
# 16. SUMMARIZATION MIDDLEWARE
# ============================================================

# Problem:
#
#     Conversation keeps growing
#
#     User
#     ↓
#     AI
#     ↓
#     User
#     ↓
#     Tool
#     ↓
#     AI
#     ↓
#     User
#     ↓
#     ...
#
# Eventually:
#
#     context window
#           ↓
#        TOO LARGE
#
#
# SummarizationMiddleware automatically summarizes older
# conversation history when token usage approaches a
# configured threshold.


summarized_agent = create_agent(
    model=gemini_model,

    tools=[
        get_weather,
    ],

    middleware=[
        SummarizationMiddleware(
            model=gemini_model,

            # Trigger summarization when conversation reaches
            # approximately this many tokens.
            trigger=("tokens", 4000),

            # Keep the most recent 20 messages untouched.
            keep=("messages", 20),
        )
    ],
)


# You use it normally:

result = summarized_agent.invoke(
    {
        "messages": [
            {
                "role": "user",
                "content": "Remember that I prefer concise answers.",
            }
        ]
    }
)


# You DON'T manually call:
#
#     summarize()
#
#
# Middleware watches the conversation automatically.


# ============================================================
# 17. WHAT SUMMARIZATION MIDDLEWARE ACTUALLY DOES
# ============================================================

#
# BEFORE:
#
# Message 1
# Message 2
# Message 3
# ...
# Message 100
#
# ↓ token threshold reached
#
# SummarizationMiddleware
#
# ↓
#
# OLD MESSAGES
#       ↓
#   SUMMARY
#
# RECENT MESSAGES
#       ↓
#   kept intact
#
#
# So conceptually:
#
#     [OLD 1]
#     [OLD 2]
#     [OLD 3]
#        ↓
#     SUMMARY
#        +
#     [RECENT messages]
#
#
# The model still gets the important context without
# carrying every old message forever.
#
# LangChain also takes care to preserve AI/Tool message
# relationships when summarizing.


# ============================================================
# 18. EVERYTHING IN ONE MENTAL MODEL
# ============================================================

#
#                 LANGCHAIN
#
#
#       ┌──────────────────────────┐
#       │        MODEL             │
#       │                          │
#       │ model.invoke()           │
#       │ model.bind_tools()       │
#       │ model.with_structured_   │
#       │     output(Pydantic)     │
#       └────────────┬─────────────┘
#                    │
#                    │
#                    ▼
#       ┌──────────────────────────┐
#       │        AGENT             │
#       │                          │
#       │ create_agent()           │
#       │                          │
#       │ invoke()                 │
#       │ stream()                 │
#       │ batch()                  │
#       │                          │
#       │ model ↔ tools loop       │
#       └────────────┬─────────────┘
#                    │
#          ┌─────────┴─────────┐
#          │                   │
#          ▼                   ▼
#       TOOLS                MIDDLEWARE
#                              │
#                    ┌─────────┴──────────┐
#                    │                    │
#                    ▼                    ▼
#                  HITL              SUMMARIZATION
#                    │                    │
#             approve/edit/reject     compress old
#                                    conversation
#
#
# ============================================================
# 19. THE REVISION CHEAT SHEET
# ============================================================

#
# MODEL
#
# model.invoke(...)
#     → direct LLM call
#
#
# MODEL + STRUCTURED OUTPUT
#
# model.with_structured_output(MyPydanticClass)
#     → model returns structured Pydantic data
#
#
# MODEL + TOOLS
#
# model.bind_tools([tools])
#     → model can REQUEST tools
#     → YOU handle execution/loop
#
#
# AGENT
#
# create_agent(
#     model=model,
#     tools=[tools]
# )
#     → model + automatic tool execution loop
#
#
# AGENT + STRUCTURED OUTPUT
#
# create_agent(
#     model=model,
#     tools=[tools],
#     response_format=MyPydanticClass
# )
#     → agent runs tools
#     → final structured response
#     → result["structured_response"]
#
#
# AGENT INVOKE
#
# agent.invoke(input)
#     → one complete run
#
#
# AGENT STREAM
#
# agent.stream(input)
#     → stream execution/results
#
#
# AGENT BATCH
#
# agent.batch([input1, input2, input3])
#     → execute multiple inputs
#
#
# HITL
#
# HumanInTheLoopMiddleware(...)
#     → pause before selected tools
#
# approve
#     → execute original action
#
# edit
#     → execute modified action
#
# reject
#     → don't execute + give feedback to model
#
#
# SUMMARIZATION
#
# SummarizationMiddleware(...)
#     → automatically compress old conversation
#     → preserve recent messages
#     → control context growth
#
#
# ============================================================
# 20. THE ONE-LINE MEMORY TRICK
# ============================================================

#
# MODEL:
#
#     "Give me an answer."
#
# bind_tools:
#
#     "You may ASK for a tool."
#
# AGENT:
#
#     "I'll RUN the tools and keep looping."
#
# STRUCTURED OUTPUT:
#
#     "Give me the answer in THIS schema."
#
# STREAM:
#
#     "Show me the execution as it happens."
#
# BATCH:
#
#     "Do this for MANY inputs."
#
# HITL:
#
#     "STOP before dangerous tools and ASK A HUMAN."
#
# SUMMARIZATION:
#
#     "The conversation is getting too big; COMPRESS OLD
#      HISTORY while keeping recent context."
#
# ============================================================